# BPE Tokenizer — Deep Dive

This notebook walks through **every function** of `code/bpe_tokenizer.py`, from raw bytes to trained vocab.  
Run cells top-to-bottom; each section is self-contained but builds on prior state.

**Sections:**
1. Setup & imports
2. `bytes_to_unicode` — the byte encoder
3. Pre-tokenization regex
4. `_chunk_key` — text → base symbols
5. `_merge_symbols` — the atomic merge op
6. Stage A in slow-motion (tiny corpus)
7. `_bpe` — heap+linked-list encoder
8. Stage B bigram training
9. Full `encode` → `decode` roundtrip
10. `encode_with_offsets` — NER alignment
11. Load a real trained tokenizer & inspect it
12. Vocabulary statistics & visualisations

---
## 1 — Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'code'))

import heapq
from collections import Counter
import regex as re
import pickle

# Import the live module so edits to bpe_tokenizer.py are reflected after kernel restart
import importlib
import bpe_tokenizer as _mod
importlib.reload(_mod)

from bpe_tokenizer import (
    BPETokenizer, bytes_to_unicode, get_pairs,
    PRETOKENIZE_PATTERN, PRETOKENIZE_PATTERN_SOCIAL
)

print('Python:', sys.version)
print('OK')

Python: 3.11.14 (main, Dec 17 2025, 21:07:37) [Clang 21.1.4 ]
OK


---
## 2 — `bytes_to_unicode`: why every byte gets a printable char

BPE works on **strings**, not bytes.  But raw UTF-8 bytes include control chars (0x00-0x1F) that would silently break string comparison and printing.  
The trick (from GPT-2): **map every byte 0-255 to a printable unicode character** so the byte sequence is a normal string.

> Bytes already in the printable ASCII / Latin-1 ranges keep their natural characters.  
> The remaining 68 bytes get mapped to the block U+0100–U+0143 (never used in normal text).

In [2]:
b2u = bytes_to_unicode()   # dict: int(byte) -> str(unicode char)

print(f'Total entries: {len(b2u)}  (one per byte value 0-255)')
print()

# Show a few interesting cases
interesting_bytes = [
    (0,    'NUL — control, gets remapped'),
    (32,   'SPACE'),
    (65,   'A — already printable ASCII'),
    (9,    'TAB — control, gets remapped'),
    (195,  'first byte of ü in UTF-8'),
    (188,  'some Latin-1 byte'),
]
for byte_val, note in interesting_bytes:
    char = b2u[byte_val]
    print(f'  byte {byte_val:3d} (0x{byte_val:02X})  ->  repr={repr(char)}  ord={ord(char)}   # {note}')

Total entries: 256  (one per byte value 0-255)

  byte   0 (0x00)  ->  repr='Ā'  ord=256   # NUL — control, gets remapped
  byte  32 (0x20)  ->  repr='Ġ'  ord=288   # SPACE
  byte  65 (0x41)  ->  repr='A'  ord=65   # A — already printable ASCII
  byte   9 (0x09)  ->  repr='ĉ'  ord=265   # TAB — control, gets remapped
  byte 195 (0xC3)  ->  repr='Ã'  ord=195   # first byte of ü in UTF-8
  byte 188 (0xBC)  ->  repr='¼'  ord=188   # some Latin-1 byte


In [3]:
# The SPACE byte (32) maps to Ġ (U+0120) — the famous GPT-2 space marker.
# This is what BPETokenizer.space_token is.
space_char = b2u[32]
print(f'space_token = {repr(space_char)}  (U+{ord(space_char):04X})')

# Verify invertibility: decode the whole map back to bytes
u2b = {v: k for k, v in b2u.items()}
assert all(u2b[b2u[i]] == i for i in range(256)), 'not bijective!'
print('Bijective: every byte encodes/decodes cleanly ✓')

space_token = 'Ġ'  (U+0120)
Bijective: every byte encodes/decodes cleanly ✓


In [4]:
# Encode an arbitrary string to its byte-level token sequence
def show_byte_encode(text, b2u=b2u):
    raw = text.encode('utf-8')
    symbols = [b2u[b] for b in raw]
    print(f'Input:   {repr(text)}')
    print(f'UTF-8:   {list(raw)}')
    print(f'Symbols: {symbols}')
    print(f'(joined as string: {repr("".join(symbols))})')
    return symbols

show_byte_encode('Hello')
print()
show_byte_encode(' Hello')   # leading space -> Ġ prefix
print()
show_byte_encode('naïve')    # multi-byte Unicode

Input:   'Hello'
UTF-8:   [72, 101, 108, 108, 111]
Symbols: ['H', 'e', 'l', 'l', 'o']
(joined as string: 'Hello')

Input:   ' Hello'
UTF-8:   [32, 72, 101, 108, 108, 111]
Symbols: ['Ġ', 'H', 'e', 'l', 'l', 'o']
(joined as string: 'ĠHello')

Input:   'naïve'
UTF-8:   [110, 97, 195, 175, 118, 101]
Symbols: ['n', 'a', 'Ã', '¯', 'v', 'e']
(joined as string: 'naÃ¯ve')


['n', 'a', 'Ã', '¯', 'v', 'e']

---
## 3 — Pre-tokenization: the regex

Before BPE, text is split into **chunks** (roughly: words, punctuation runs, spaces).  
This is crucial: BPE merges **within** a chunk, never across chunk boundaries — so `"don't"` never accidentally merges the `n` of `don` with the `'` of `'t`.

In [5]:
pat = re.compile(PRETOKENIZE_PATTERN, re.UNICODE)
pat_social = re.compile(PRETOKENIZE_PATTERN_SOCIAL, re.UNICODE)

def show_pretokenize(text, pattern=pat):
    chunks = pattern.findall(text)
    print(f'Input: {repr(text)}')
    for i, c in enumerate(chunks):
        print(f'  chunk[{i}] = {repr(c)}')
    return chunks

print('=== Standard pattern ===')
show_pretokenize("Hello, world! Don't stop.")
print()
show_pretokenize("naïve café résumé")
print()
show_pretokenize("Price: $1,234.56")

=== Standard pattern ===
Input: "Hello, world! Don't stop."
  chunk[0] = 'Hello'
  chunk[1] = ','
  chunk[2] = ' world'
  chunk[3] = '!'
  chunk[4] = ' Don'
  chunk[5] = "'t"
  chunk[6] = ' stop'
  chunk[7] = '.'

Input: 'naïve café résumé'
  chunk[0] = 'naïve'
  chunk[1] = ' café'
  chunk[2] = ' résumé'

Input: 'Price: $1,234.56'
  chunk[0] = 'Price'
  chunk[1] = ':'
  chunk[2] = ' $'
  chunk[3] = '1'
  chunk[4] = ','
  chunk[5] = '234'
  chunk[6] = '.'
  chunk[7] = '56'


['Price', ':', ' $', '1', ',', '234', '.', '56']

In [6]:
print('=== Social media pattern (adds @handle and #hashtag rules) ===')
tweet = "@elonmusk #AI is wild lol I can't believe it!!"

print('Standard:')
show_pretokenize(tweet, pat)
print()
print('Social:')
show_pretokenize(tweet, pat_social)

=== Social media pattern (adds @handle and #hashtag rules) ===
Standard:
Input: "@elonmusk #AI is wild lol I can't believe it!!"
  chunk[0] = '@elonmusk'
  chunk[1] = ' #'
  chunk[2] = 'AI'
  chunk[3] = ' is'
  chunk[4] = ' wild'
  chunk[5] = ' lol'
  chunk[6] = ' I'
  chunk[7] = ' can'
  chunk[8] = "'t"
  chunk[9] = ' believe'
  chunk[10] = ' it'
  chunk[11] = '!!'

Social:
Input: "@elonmusk #AI is wild lol I can't believe it!!"
  chunk[0] = '@elonmusk'
  chunk[1] = ' #'
  chunk[2] = 'AI'
  chunk[3] = ' is'
  chunk[4] = ' wild'
  chunk[5] = ' lol'
  chunk[6] = ' I'
  chunk[7] = ' can'
  chunk[8] = "'t"
  chunk[9] = ' believe'
  chunk[10] = ' it'
  chunk[11] = '!!'


['@elonmusk',
 ' #',
 'AI',
 ' is',
 ' wild',
 ' lol',
 ' I',
 ' can',
 "'t",
 ' believe',
 ' it',
 '!!']

In [7]:
# Numbers: the pattern groups digits in runs of 1-3
# This prevents "123456789" from becoming one giant token
print('Number grouping (≤3 digits per chunk):')
show_pretokenize("Call 1-800-555-1234 for details.")

Number grouping (≤3 digits per chunk):
Input: 'Call 1-800-555-1234 for details.'
  chunk[0] = 'Call'
  chunk[1] = ' '
  chunk[2] = '1'
  chunk[3] = '-'
  chunk[4] = '800'
  chunk[5] = '-'
  chunk[6] = '555'
  chunk[7] = '-'
  chunk[8] = '123'
  chunk[9] = '4'
  chunk[10] = ' for'
  chunk[11] = ' details'
  chunk[12] = '.'


['Call',
 ' ',
 '1',
 '-',
 '800',
 '-',
 '555',
 '-',
 '123',
 '4',
 ' for',
 ' details',
 '.']

---
## 4 — `_chunk_key`: text chunk → base symbol tuple

Each chunk string is converted to a **tuple of byte-encoded symbols** — the starting state for BPE merges.

In [8]:
tok_scratch = BPETokenizer(vocab_size=500)  # tiny, just for method access

def show_chunk_key(chunk):
    key = tok_scratch._chunk_key(chunk)
    print(f'chunk={repr(chunk)}')
    print(f'  key = {key}')
    # Decode back to show what each symbol represents
    decoded = [bytearray([tok_scratch.byte_decoder[c]]).decode('utf-8', errors='replace') for c in key]
    print(f'  decoded symbols: {decoded}')
    return key

show_chunk_key('Hello')
print()
show_chunk_key(' world')   # leading Ġ from the space byte
print()
show_chunk_key("n't")

chunk='Hello'
  key = ('H', 'e', 'l', 'l', 'o')
  decoded symbols: ['H', 'e', 'l', 'l', 'o']

chunk=' world'
  key = ('Ġ', 'w', 'o', 'r', 'l', 'd')
  decoded symbols: [' ', 'w', 'o', 'r', 'l', 'd']

chunk="n't"
  key = ('n', "'", 't')
  decoded symbols: ['n', "'", 't']


('n', "'", 't')

---
## 5 — `_merge_symbols`: the atomic merge operation

Given a symbol list and a pair `(a, b)`, replace every adjacent `a b` occurrence with the merged symbol `ab`.

In [9]:
# Walk through a simple example
symbols = ['H', 'e', 'l', 'l', 'o']
print('Before:', symbols)

step1 = BPETokenizer._merge_symbols(symbols, ('l', 'l'))
print('Merge (l,l):', step1)

step2 = BPETokenizer._merge_symbols(step1, ('H', 'e'))
print('Merge (H,e):', step2)

step3 = BPETokenizer._merge_symbols(step2, ('He', 'll'))
print('Merge (He,ll):', step3)

Before: ['H', 'e', 'l', 'l', 'o']
Merge (l,l): ['H', 'e', 'll', 'o']
Merge (H,e): ['He', 'll', 'o']
Merge (He,ll): ['Hell', 'o']


In [10]:
# Edge cases
# Pair at end of list
print(BPETokenizer._merge_symbols(['a', 'b', 'c', 'a', 'b'], ('a', 'b')))
# Overlapping: 'a a a' — only non-overlapping left-to-right merges happen
print(BPETokenizer._merge_symbols(['a', 'a', 'a'], ('a', 'a')))

['ab', 'c', 'ab']
['aa', 'a']


---
## 6 — Stage A in slow-motion: training within-word BPE

Train a tiny tokenizer on a handful of sentences so we can **watch every merge** happen.

In [11]:
# Monkey-patch _train_within_word to print each merge as it happens

import heapq
from collections import Counter

def train_within_word_verbose(self, line_counts, budget, max_show=30):
    """Identical logic to the real _train_within_word, but prints each merge."""
    word_freqs = Counter()
    line_keys = {}
    for line, freq in line_counts.items():
        keys = [self._chunk_key(chunk) for chunk in self.pat.findall(line)]
        line_keys[line] = keys
        for k in keys:
            word_freqs[k] += freq

    word_syms = [list(k) for k in word_freqs.keys()]
    word_freq = list(word_freqs.values())

    pair_freqs = {}
    pair_to_words = {}
    for idx, syms in enumerate(word_syms):
        f = word_freq[idx]
        for p in zip(syms, syms[1:]):
            pair_freqs[p] = pair_freqs.get(p, 0) + f
            pair_to_words.setdefault(p, set()).add(idx)

    lbpe_exp = getattr(self, '_lbpe_exp', 0.0)
    def _score(p, f):
        return f * (len(p[0]) + len(p[1])) ** lbpe_exp

    heap = [(-_score(p, f), p) for p, f in pair_freqs.items()]
    heapq.heapify(heap)

    def push(p):
        f = pair_freqs.get(p, 0)
        if f > 0:
            heapq.heappush(heap, (-_score(p, f), p))

    rank = 0
    shown = 0
    while len(self.token_to_id) < budget and heap:
        neg_sc, best = heapq.heappop(heap)
        if _score(best, pair_freqs.get(best, 0)) != -neg_sc:
            continue
        if pair_freqs.get(best, 0) < self._min_pair_freq:
            break

        # Decode the pair to readable form
        def sym_readable(s):
            return bytearray(self.byte_decoder[c] for c in s).decode('utf-8', errors='replace')

        freq = pair_freqs[best]
        if shown < max_show:
            merged_surface = sym_readable(best[0] + best[1])
            a_surf = sym_readable(best[0])
            b_surf = sym_readable(best[1])
            print(f'  rank={rank:3d}  freq={freq:4d}  ({repr(a_surf)} + {repr(b_surf)})  ->  {repr(merged_surface)}')
            shown += 1
        elif shown == max_show:
            print(f'  ... (suppressing remaining merges)')
            shown += 1

        self.bpe_ranks[best] = rank
        rank += 1
        self._add_token(best[0] + best[1])

        for idx in list(pair_to_words.get(best, ())):
            syms = word_syms[idx]
            f = word_freq[idx]
            old = Counter(zip(syms, syms[1:]))
            new_syms = self._merge_symbols(syms, best)
            new = Counter(zip(new_syms, new_syms[1:]))
            word_syms[idx] = new_syms
            for p, c in old.items():
                pair_freqs[p] = pair_freqs.get(p, 0) - f * c
                if pair_freqs[p] <= 0:
                    pair_freqs.pop(p, None)
                if p not in new:
                    s = pair_to_words.get(p)
                    if s is not None:
                        s.discard(idx)
                if p != best:
                    push(p)
            for p, c in new.items():
                pair_freqs[p] = pair_freqs.get(p, 0) + f * c
                pair_to_words.setdefault(p, set()).add(idx)
                push(p)

        pair_freqs.pop(best, None)
        pair_to_words.pop(best, None)

    self._atomic = {}
    for idx, key in enumerate(word_freqs):
        if len(word_syms[idx]) == 1:
            self._atomic[key] = word_syms[idx][0]

    print(f'\nStage A done: {rank} merges, vocab size = {len(self.token_to_id)}')
    return line_keys

In [12]:
SMALL_CORPUS = [
    "the cat sat on the mat",
    "the cat sat on the hat",
    "the cat ate the rat",
    "a fat cat on a flat mat",
    "cats and hats and bats",
    "the cat is the best cat",
] * 10  # repeat 10x so frequencies are meaningful

tiny = BPETokenizer(vocab_size=300)
tiny._auto_configure(SMALL_CORPUS)
tiny._min_pair_freq = 2

# Add base vocab (256 bytes)
for char in tiny.byte_encoder.values():
    tiny._add_token(char)

line_counts = Counter(t for t in SMALL_CORPUS if t.strip())

print('=== Stage A merges (within-word BPE) ===')
line_keys = train_within_word_verbose(tiny, line_counts, budget=290)

=== Stage A merges (within-word BPE) ===
  rank=  0  freq= 180  ('a' + 't')  ->  'at'
  rank=  1  freq=  80  ('h' + 'e')  ->  'he'
  rank=  2  freq=  80  ('t' + 'he')  ->  'the'
  rank=  3  freq=  70  ('c' + 'at')  ->  'cat'
  rank=  4  freq=  60  (' ' + 'cat')  ->  ' cat'
  rank=  5  freq=  40  (' ' + 'the')  ->  ' the'
  rank=  6  freq=  30  ('o' + 'n')  ->  'on'
  rank=  7  freq=  30  (' ' + 'a')  ->  ' a'
  rank=  8  freq=  30  (' ' + 'on')  ->  ' on'
  rank=  9  freq=  20  ('at' + 's')  ->  'ats'
  rank= 10  freq=  20  ('m' + 'at')  ->  'mat'
  rank= 11  freq=  20  ('n' + 'd')  ->  'nd'
  rank= 12  freq=  20  ('s' + 'at')  ->  'sat'
  rank= 13  freq=  20  (' ' + 'b')  ->  ' b'
  rank= 14  freq=  20  (' ' + 'f')  ->  ' f'
  rank= 15  freq=  20  (' ' + 'h')  ->  ' h'
  rank= 16  freq=  20  (' ' + 'mat')  ->  ' mat'
  rank= 17  freq=  20  (' ' + 'sat')  ->  ' sat'
  rank= 18  freq=  20  (' a' + 'nd')  ->  ' and'
  rank= 19  freq=  10  ('at' + 'e')  ->  'ate'
  rank= 20  freq=  10  ('

In [13]:
# Inspect the atomic words that Stage A produced (each compressed to 1 token)
print(f'Atomic words (whole word -> single token): {len(tiny._atomic)}')
print()
for key, tok in list(tiny._atomic.items())[:20]:
    surface = bytearray(tiny.byte_decoder[c] for c in tok).decode('utf-8', errors='replace')
    original = bytearray(tiny.byte_decoder[c] for c in key).decode('utf-8', errors='replace')
    print(f'  {repr(original):15s} -> token {repr(tok):20s}  (surface: {repr(surface)})')

Atomic words (whole word -> single token): 14

  'the'           -> token 'the'                 (surface: 'the')
  ' cat'          -> token 'Ġcat'                (surface: ' cat')
  ' sat'          -> token 'Ġsat'                (surface: ' sat')
  ' on'           -> token 'Ġon'                 (surface: ' on')
  ' the'          -> token 'Ġthe'                (surface: ' the')
  ' mat'          -> token 'Ġmat'                (surface: ' mat')
  ' ate'          -> token 'Ġate'                (surface: ' ate')
  ' rat'          -> token 'Ġrat'                (surface: ' rat')
  'a'             -> token 'a'                   (surface: 'a')
  ' a'            -> token 'Ġa'                  (surface: ' a')
  'cats'          -> token 'cats'                (surface: 'cats')
  ' and'          -> token 'Ġand'                (surface: ' and')
  ' bats'         -> token 'Ġbats'               (surface: ' bats')
  ' is'           -> token 'Ġis'                 (surface: ' is')


---
## 7 — `_bpe`: the heap + doubly-linked-list encoder

At **inference time**, `_bpe` applies the learned merges to a symbol list.  
It uses a **min-heap** keyed by merge rank so the lowest-rank (earliest-learned) pair is always merged first — O(n log n) vs the naive O(n²) GPT-2 approach.

In [14]:
# Patch _bpe to print each merge step
def _bpe_verbose(self, symbols):
    n = len(symbols)
    if n < 2:
        return list(symbols)

    syms = list(symbols)
    prev = list(range(-1, n - 1))
    nxt  = list(range(1, n + 1))
    alive = [True] * n
    ranks = self.bpe_ranks
    INF = float('inf')

    heap = []
    for i in range(n - 1):
        r = ranks.get((syms[i], syms[i+1]), INF)
        if r < INF:
            heapq.heappush(heap, (r, i))

    step = 0
    while heap:
        r, i = heapq.heappop(heap)
        if not alive[i]: continue
        j = nxt[i]
        if j >= n or not alive[j]: continue
        if ranks.get((syms[i], syms[j]), INF) != r: continue

        a, b = syms[i], syms[j]
        syms[i] = a + b
        alive[j] = False
        nxt[i] = nxt[j]
        if nxt[j] < n:
            prev[nxt[j]] = i

        current = [syms[k] for k in range(n) if alive[k]]

        def decode_sym(s):
            return bytearray(self.byte_decoder[c] for c in s).decode('utf-8', errors='replace')

        print(f'  step {step}: merged rank={r} ({repr(decode_sym(a))} + {repr(decode_sym(b))}) -> {repr(decode_sym(syms[i]))}')
        print(f'           state: {[decode_sym(s) for s in current]}')
        step += 1

        p = prev[i]
        if p >= 0:
            r2 = ranks.get((syms[p], syms[i]), INF)
            if r2 < INF: heapq.heappush(heap, (r2, p))
        k = nxt[i]
        if k < n:
            r2 = ranks.get((syms[i], syms[k]), INF)
            if r2 < INF: heapq.heappush(heap, (r2, i))

    return [syms[i] for i in range(n) if alive[i]]

In [15]:
# Use the tiny tokenizer trained above to watch a single chunk get BPE'd

chunk = 'cats'
base_symbols = [tiny.byte_encoder[b] for b in chunk.encode('utf-8')]
print(f'Encoding chunk: {repr(chunk)}')
print(f'Base symbols:   {base_symbols}')
print(f'Readable:       {[chr(tiny.byte_decoder[s]) if tiny.byte_decoder[s] < 128 else "?" for s in base_symbols]}')
print()
print('--- BPE merge steps ---')
result = _bpe_verbose(tiny, base_symbols)
print()
print('Final symbols:', result)

Encoding chunk: 'cats'
Base symbols:   ['c', 'a', 't', 's']
Readable:       ['c', 'a', 't', 's']

--- BPE merge steps ---
  step 0: merged rank=0 ('a' + 't') -> 'at'
           state: ['c', 'at', 's']
  step 1: merged rank=3 ('c' + 'at') -> 'cat'
           state: ['cat', 's']
  step 2: merged rank=20 ('cat' + 's') -> 'cats'
           state: ['cats']

Final symbols: ['cats']


In [16]:
# Try a word NOT in training data — BPE degrades gracefully
chunk = 'zzz'
base_symbols = [tiny.byte_encoder[b] for b in chunk.encode('utf-8')]
print(f'Encoding unseen chunk: {repr(chunk)}')
print(f'Base symbols: {base_symbols}  (no known merges -> stays as individual bytes)')
result = _bpe_verbose(tiny, base_symbols)
print('Final:', result)

Encoding unseen chunk: 'zzz'
Base symbols: ['z', 'z', 'z']  (no known merges -> stays as individual bytes)
Final: ['z', 'z', 'z']


---
## 8 — Stage B: bigram training

After Stage A, some whole words are **atomic** (compressed to one token).  
Stage B counts how often adjacent atomic words appear together and creates **cross-word bigram tokens**.

In [17]:
# Show which atoms are eligible for bigram merges (must contain a letter)
word_atoms = {
    key: tok
    for key, tok in tiny._atomic.items()
    if any(c.isalpha() for c in bytearray(tiny.byte_decoder[c] for c in tok).decode('utf-8', errors='replace'))
}

print(f'Total atomic words: {len(tiny._atomic)}')
print(f'Letter-containing (eligible for bigrams): {len(word_atoms)}')
print()

# Show what pairs actually appear adjacent in the corpus
bigram_freqs = Counter()
for line, freq in line_counts.items():
    atoms = [word_atoms.get(k) for k in line_keys[line]]
    for a, b in zip(atoms, atoms[1:]):
        if a is not None and b is not None:
            bigram_freqs[(a, b)] += freq

print(f'Distinct adjacent atom pairs: {len(bigram_freqs)}')
print()
print('Top bigrams (raw token pair -> surface form -> frequency):')
for (a, b), freq in bigram_freqs.most_common(10):
    def surf(t):
        return bytearray(tiny.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
    print(f'  {repr(surf(a)):10s} + {repr(surf(b)):10s}  ->  {repr(surf(a+b)):15s}  freq={freq}')

Total atomic words: 14
Letter-containing (eligible for bigrams): 14

Distinct adjacent atom pairs: 14

Top bigrams (raw token pair -> surface form -> frequency):
  'the'      + ' cat'      ->  'the cat'        freq=40
  ' cat'     + ' sat'      ->  ' cat sat'       freq=20
  ' sat'     + ' on'       ->  ' sat on'        freq=20
  ' on'      + ' the'      ->  ' on the'        freq=20
  ' the'     + ' mat'      ->  ' the mat'       freq=10
  ' cat'     + ' ate'      ->  ' cat ate'       freq=10
  ' ate'     + ' the'      ->  ' ate the'       freq=10
  ' the'     + ' rat'      ->  ' the rat'       freq=10
  ' cat'     + ' on'       ->  ' cat on'        freq=10
  ' on'      + ' a'        ->  ' on a'          freq=10


In [18]:
# Now run the actual Stage B on the tiny tokenizer
tiny._train_bigrams(line_counts, line_keys)

print(f'Bigram merges learned: {len(tiny.bigram_merges)}')
print()
print('Learned bigrams (surface form, frequency):')
for surface, freq in tiny.get_bigrams()[:15]:
    print(f'  {repr(surface):20s}  freq={freq}')

Bigram merges learned: 10

Learned bigrams (surface form, frequency):
  'the cat'             freq=40
  'cat sat'             freq=20
  'sat on'              freq=20
  'on the'              freq=20
  'the mat'             freq=10
  'cat ate'             freq=10
  'ate the'             freq=10
  'the rat'             freq=10
  'cat on'              freq=10
  'on a'                freq=10


---
## 9 — Full `encode` → `decode` roundtrip

In [19]:
# Build a complete small tokenizer the normal way
mini = BPETokenizer(vocab_size=400)
mini.train(SMALL_CORPUS)

print(f'Vocab size: {mini.get_vocab_size()}')
print(f'Stage-A merges: {len(mini.bpe_ranks)}')
print(f'Stage-B bigrams: {len(mini.bigram_merges)}')

Vocab size: 318
Stage-A merges: 35
Stage-B bigrams: 23


In [20]:
def show_encode(tok, text):
    ids = tok.encode(text)
    tokens = [tok.id_to_token[i] for i in ids]

    def surf(t):
        if t in tok.special_tokens:
            return t
        try:
            return bytearray(tok.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
        except KeyError:
            return t

    print(f'Input:   {repr(text)}')
    print(f'IDs:     {ids}')
    print(f'Tokens:  {tokens}')
    print(f'Surface: {[surf(t) for t in tokens]}')
    decoded = tok.decode(ids)
    print(f'Decoded: {repr(decoded)}')
    print(f'Lossless: {text == decoded}')
    return ids

print('=== In-vocab sentence ===')
show_encode(mini, 'the cat sat on the mat')
print()
print('=== Out-of-vocab words ===')
show_encode(mini, 'the quantum elephant dances')

=== In-vocab sentence ===
Input:   'the cat sat on the mat'
IDs:     [295, 297, 299]
Tokens:  ['theĠcat', 'ĠsatĠon', 'ĠtheĠmat']
Surface: ['the cat', ' sat on', ' the mat']
Decoded: 'the cat sat on the mat'
Lossless: True

=== Out-of-vocab words ===
Input:   'the quantum elephant dances'
IDs:     [262, 224, 84, 88, 68, 81, 87, 88, 80, 224, 72, 79, 72, 83, 75, 68, 81, 87, 224, 71, 68, 81, 70, 281]
Tokens:  ['the', 'Ġ', 'q', 'u', 'a', 'n', 't', 'u', 'm', 'Ġ', 'e', 'l', 'e', 'p', 'h', 'a', 'n', 't', 'Ġ', 'd', 'a', 'n', 'c', 'es']
Surface: ['the', ' ', 'q', 'u', 'a', 'n', 't', 'u', 'm', ' ', 'e', 'l', 'e', 'p', 'h', 'a', 'n', 't', ' ', 'd', 'a', 'n', 'c', 'es']
Decoded: 'the quantum elephant dances'
Lossless: True


[262,
 224,
 84,
 88,
 68,
 81,
 87,
 88,
 80,
 224,
 72,
 79,
 72,
 83,
 75,
 68,
 81,
 87,
 224,
 71,
 68,
 81,
 70,
 281]

In [21]:
# Demonstrate Stage-B bigram firing during encode
# (Two adjacent atomic words get merged into one bigram token)

print('Learned bigrams:', mini.get_bigrams()[:5])
print()

# Construct a sentence guaranteed to contain the top bigram
if mini.bigram_merges:
    top_pair = list(mini.bigram_merges.keys())[0]
    def surf(t):
        return bytearray(mini.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
    word_a = surf(top_pair[0]).strip()
    word_b = surf(top_pair[1]).strip()
    sentence = f'{word_a} {word_b}'
    print(f'Bigram pair surface: "{word_a}" + "{word_b}"  ->  expected single token')
    ids = show_encode(mini, sentence)
    print()
    bigram_tok = mini.bigram_merges[top_pair]
    bigram_id  = mini.token_to_id[bigram_tok]
    print(f'Bigram token id: {bigram_id}  (present in output: {bigram_id in ids})')

Learned bigrams: [('the cat', 40), ('cat sat', 20), ('sat on', 20), ('on the', 20), ('the mat', 10)]

Bigram pair surface: "the" + "cat"  ->  expected single token
Input:   'the cat'
IDs:     [295]
Tokens:  ['theĠcat']
Surface: ['the cat']
Decoded: 'the cat'
Lossless: True

Bigram token id: 295  (present in output: True)


---
## 10 — `encode_with_offsets`: NER alignment

NER needs to know which character span each token came from.  
`encode_with_offsets` returns `(token_ids, [(start, end), ...])` where offsets are **character positions** in the original string.

In [22]:
text = 'the fat cat sat'
ids, offsets = mini.encode_with_offsets(text)

print(f'Input: {repr(text)}')
print()
print(f'{"Token ID":>10}  {"Token surface":>20}  {"Start":>6}  {"End":>4}  Span')
print('-' * 60)
for tid, (start, end) in zip(ids, offsets):
    token = mini.id_to_token.get(tid, '[UNK]')
    try:
        surface = bytearray(mini.byte_decoder[c] for c in token).decode('utf-8', errors='replace')
    except (KeyError, TypeError):
        surface = token
    span_text = text[start:end]
    print(f'{tid:>10}  {repr(surface):>20}  {start:>6}  {end:>4}  {repr(span_text)}')

Input: 'the fat cat sat'

  Token ID         Token surface   Start   End  Span
------------------------------------------------------------
       262                 'the'       0     3  'the'
       305            ' fat cat'       3    11  ' fat cat'
       277                ' sat'      11    15  ' sat'


In [23]:
# Verify: encode_with_offsets and encode produce the same IDs
ids_a = mini.encode(text)
ids_b, _ = mini.encode_with_offsets(text)
print('IDs match:', ids_a == ids_b)

IDs match: True


In [24]:
# Show how first-subtoken labeling works for NER
# (multi-byte chars, or long words, split into multiple subtokens)

text = 'quantum cats'
ids, offsets = mini.encode_with_offsets(text)

# Simulate NER word-to-token alignment
words = text.split()
word_spans = []
pos = 0
for w in words:
    start = text.index(w, pos)
    word_spans.append((start, start + len(w), w))
    pos = start + len(w)

print('Words:', word_spans)
print()
print('Token -> Word assignment (first-subtoken gets label, rest get -100):')
labels = []
for i, (tid, (ts, te)) in enumerate(zip(ids, offsets)):
    token = mini.id_to_token.get(tid, '?')
    for ws, we, word in word_spans:
        if ts >= ws and te <= we:
            if ts == ws:  # first subtoken of this word
                label = f'B-{word.upper()}'  # fake NER label for demo
                is_first = True
            else:
                label = '-100 (ignored)'
                is_first = False
            print(f'  token[{i}] id={tid:4d} span=({ts},{te}) word={repr(word):10s} label={label}')
            break

Words: [(0, 7, 'quantum'), (8, 12, 'cats')]

Token -> Word assignment (first-subtoken gets label, rest get -100):
  token[0] id=  84 span=(0,1) word='quantum'  label=B-QUANTUM
  token[1] id=  88 span=(1,2) word='quantum'  label=-100 (ignored)
  token[2] id=  68 span=(2,3) word='quantum'  label=-100 (ignored)
  token[3] id=  81 span=(3,4) word='quantum'  label=-100 (ignored)
  token[4] id=  87 span=(4,5) word='quantum'  label=-100 (ignored)
  token[5] id=  88 span=(5,6) word='quantum'  label=-100 (ignored)
  token[6] id=  80 span=(6,7) word='quantum'  label=-100 (ignored)
  token[8] id=  86 span=(11,12) word='cats'     label=-100 (ignored)


---
## 11 — Load a real trained tokenizer & inspect it

In [25]:
from base_tokenizer import BaseTokenizer

tok1 = BaseTokenizer.load('trained_tokenizers/tokenizer_1.pkl')  # social media
tok2 = BaseTokenizer.load('trained_tokenizers/tokenizer_2.pkl')  # formal/news

for name, tok in [('tokenizer_1 (social)', tok1), ('tokenizer_2 (news)', tok2)]:
    print(f'--- {name} ---')
    print(f'  Vocab size:      {tok.get_vocab_size()}')
    print(f'  Stage-A merges:  {len(tok.bpe_ranks)}')
    print(f'  Stage-B bigrams: {len(tok.bigram_merges)}')
    print(f'  space_token:     {repr(tok.space_token)}')
    print()

--- tokenizer_1 (social) ---
  Vocab size:      5000
  Stage-A merges:  4340
  Stage-B bigrams: 400
  space_token:     'Ġ'

--- tokenizer_2 (news) ---
  Vocab size:      5000
  Stage-A merges:  4340
  Stage-B bigrams: 400
  space_token:     'Ġ'



In [26]:
# Top Stage-A merges by rank (rank 0 = first merged = most frequent)
def top_stageA_merges(tok, n=20):
    by_rank = sorted(tok.bpe_ranks.items(), key=lambda kv: kv[1])
    def surf(t):
        return bytearray(tok.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
    return [(rank, surf(a), surf(b), surf(a+b)) for (a,b), rank in by_rank[:n]]

print('=== tokenizer_1 top Stage-A merges ===')
for rank, a, b, merged in top_stageA_merges(tok1):
    print(f'  rank {rank:3d}: {repr(a)} + {repr(b)} -> {repr(merged)}')

print()
print('=== tokenizer_2 top Stage-A merges ===')
for rank, a, b, merged in top_stageA_merges(tok2):
    print(f'  rank {rank:3d}: {repr(a)} + {repr(b)} -> {repr(merged)}')

=== tokenizer_1 top Stage-A merges ===
  rank   0: ' ' + 't' -> ' t'
  rank   1: 'i' + 'n' -> 'in'
  rank   2: ' ' + 'a' -> ' a'
  rank   3: 'h' + 'e' -> 'he'
  rank   4: ' ' + 's' -> ' s'
  rank   5: ' ' + 'w' -> ' w'
  rank   6: ' ' + 'm' -> ' m'
  rank   7: 'r' + 'e' -> 're'
  rank   8: 'o' + 'n' -> 'on'
  rank   9: 'o' + 'u' -> 'ou'
  rank  10: ' ' + 'i' -> ' i'
  rank  11: 'h' + 'a' -> 'ha'
  rank  12: 'in' + 'g' -> 'ing'
  rank  13: ' ' + 'b' -> ' b'
  rank  14: '.' + '.' -> '..'
  rank  15: ' t' + 'o' -> ' to'
  rank  16: 'e' + 'r' -> 'er'
  rank  17: ' ' + 'f' -> ' f'
  rank  18: 'o' + 'r' -> 'or'
  rank  19: 'l' + 'l' -> 'll'

=== tokenizer_2 top Stage-A merges ===
  rank   0: ' ' + 't' -> ' t'
  rank   1: 'i' + 'n' -> 'in'
  rank   2: 'h' + 'e' -> 'he'
  rank   3: ' ' + 'a' -> ' a'
  rank   4: 'r' + 'e' -> 're'
  rank   5: 'e' + 'r' -> 'er'
  rank   6: 'o' + 'n' -> 'on'
  rank   7: 'o' + 'u' -> 'ou'
  rank   8: 'a' + 't' -> 'at'
  rank   9: 'o' + 'r' -> 'or'
  rank  10: 'a' +

In [27]:
# Stage-B bigrams
print('=== tokenizer_1 top bigrams (social) ===')
for surface, freq in tok1.get_bigrams()[:15]:
    print(f'  {repr(surface):30s}  freq={freq}')

print()
print('=== tokenizer_2 top bigrams (news) ===')
for surface, freq in tok2.get_bigrams()[:15]:
    print(f'  {repr(surface):30s}  freq={freq}')

=== tokenizer_1 top bigrams (social) ===
  "I'm"                           freq=67934
  'in the'                        freq=34635
  "don't"                         freq=34189
  "it's"                          freq=33210
  "can't"                         freq=28156
  'for the'                       freq=27387
  'going to'                      freq=26527
  'to be'                         freq=23670
  'to go'                         freq=23486
  'on the'                        freq=22932
  'to the'                        freq=22923
  'have to'                       freq=21445
  'of the'                        freq=19476
  'I have'                        freq=19447
  "i'm"                           freq=19302

=== tokenizer_2 top bigrams (news) ===
  'of the'                        freq=15655
  'in the'                        freq=12915
  'to the'                        freq=6639
  'to be'                         freq=5774
  'on the'                        freq=5662
  'for the'           

In [28]:
# Encode real domain sentences
social_tweet = "@elonmusk I can't believe this!!! #AI is wild lol"
news_headline = "Federal Reserve raises interest rates by 25 basis points."

print('=== tokenizer_1 on tweet ===')
show_encode(tok1, social_tweet)
print()

print('=== tokenizer_2 on news ===')
show_encode(tok2, news_headline)

=== tokenizer_1 on tweet ===
Input:   "@elonmusk I can't believe this!!! #AI is wild lol"
IDs:     [35, 468, 268, 3462, 78, 4615, 4937, 433, 664, 585, 36, 44, 334, 265, 1378, 588]
Tokens:  ['@', 'el', 'on', 'mus', 'k', 'ĠIĠcan', "'tĠbelieve", 'Ġthis', '!!!', 'Ġ#', 'A', 'I', 'Ġis', 'Ġw', 'ild', 'Ġlol']
Surface: ['@', 'el', 'on', 'mus', 'k', ' I can', "'t believe", ' this', '!!!', ' #', 'A', 'I', ' is', ' w', 'ild', ' lol']
Decoded: "@elonmusk I can't believe this!!! #AI is wild lol"
Lossless: True

=== tokenizer_2 on news ===
Input:   'Federal Reserve raises interest rates by 25 basis points.'
IDs:     [41, 2056, 956, 3029, 1299, 2179, 2430, 458, 719, 564, 224, 2717, 3078, 279, 296, 3917, 17]
Tokens:  ['F', 'ederal', 'ĠRes', 'erve', 'Ġra', 'ises', 'Ġinterest', 'Ġr', 'ates', 'Ġby', 'Ġ', '25', 'Ġbas', 'is', 'Ġp', 'oints', '.']
Surface: ['F', 'ederal', ' Res', 'erve', ' ra', 'ises', ' interest', ' r', 'ates', ' by', ' ', '25', ' bas', 'is', ' p', 'oints', '.']
Decoded: 'Federal Reserve rai

[41,
 2056,
 956,
 3029,
 1299,
 2179,
 2430,
 458,
 719,
 564,
 224,
 2717,
 3078,
 279,
 296,
 3917,
 17]

In [29]:
# Cross-domain: feed a tweet to the news tokenizer
print('=== tokenizer_2 (news) on tweet — domain mismatch ===')
ids = show_encode(tok2, social_tweet)
unk_id = tok2.special_tokens['[UNK]']
unk_count = ids.count(unk_id)
print(f'UNK tokens: {unk_count} / {len(ids)}')
print()
# Byte fallback means 0 UNK — every character is representable via bytes
print('(Byte-level base vocab => OOV rate is always 0, splits to byte tokens instead)')

=== tokenizer_2 (news) on tweet — domain mismatch ===
Input:   "@elonmusk I can't believe this!!! #AI is wild lol"
IDs:     [35, 322, 266, 80, 349, 78, 4851, 425, 1894, 508, 4, 4, 4, 2584, 36, 44, 350, 4323, 328, 329]
Tokens:  ['@', 'el', 'on', 'm', 'us', 'k', 'ĠIĠcan', "'t", 'Ġbelieve', 'Ġthis', '!', '!', '!', 'Ġ#', 'A', 'I', 'Ġis', 'Ġwild', 'Ġl', 'ol']
Surface: ['@', 'el', 'on', 'm', 'us', 'k', ' I can', "'t", ' believe', ' this', '!', '!', '!', ' #', 'A', 'I', ' is', ' wild', ' l', 'ol']
Decoded: "@elonmusk I can't believe this!!! #AI is wild lol"
Lossless: True
UNK tokens: 0 / 20

(Byte-level base vocab => OOV rate is always 0, splits to byte tokens instead)


---
## 12 — Vocabulary statistics & visualisations

In [30]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt

def token_length_distribution(tok, title):
    """Plot histogram of token surface lengths."""
    lengths = []
    for tid, token in tok.id_to_token.items():
        if token in tok.special_tokens:
            continue
        try:
            surface = bytearray(tok.byte_decoder[c] for c in token).decode('utf-8', errors='replace')
            lengths.append(len(surface))
        except (KeyError, TypeError):
            pass
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(lengths, bins=range(1, max(lengths)+2), edgecolor='black', color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Surface character length')
    ax.set_ylabel('# tokens')
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ", "_").replace("/","_")}.png', dpi=100)
    plt.show()
    print(f'  Max length: {max(lengths)}, mean: {sum(lengths)/len(lengths):.1f}')

token_length_distribution(tok1, 'tokenizer_1 social')
token_length_distribution(tok2, 'tokenizer_2 news')

  Max length: 16, mean: 4.4
  Max length: 16, mean: 4.8


/tmp/ipykernel_274946/339171867.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
# Token compression: compare tokens/char on held-out text

def tokens_per_char(tok, text):
    ids = tok.encode(text)
    return len(ids) / len(text)

with open('data/domain_1_dev.txt') as f:
    domain1_dev = f.read()
with open('data/domain_2_dev.txt') as f:
    domain2_dev = f.read()

print('Tokens per character (lower = more compression):')
print(f'  tok1 on domain1 (matched):    {tokens_per_char(tok1, domain1_dev[:5000]):.4f}')
print(f'  tok2 on domain1 (mismatched): {tokens_per_char(tok2, domain1_dev[:5000]):.4f}')
print(f'  tok2 on domain2 (matched):    {tokens_per_char(tok2, domain2_dev[:5000]):.4f}')
print(f'  tok1 on domain2 (mismatched): {tokens_per_char(tok1, domain2_dev[:5000]):.4f}')
print()
print('A matched tokenizer should have FEWER tokens/char (better compression).')

Tokens per character (lower = more compression):
  tok1 on domain1 (matched):    0.2952
  tok2 on domain1 (mismatched): 0.3554
  tok2 on domain2 (matched):    0.2830
  tok1 on domain2 (mismatched): 0.3446

A matched tokenizer should have FEWER tokens/char (better compression).


In [32]:
# Show the first N tokens in the vocabulary (base + first merges)
def show_vocab_slice(tok, start=0, end=280):
    print(f'Tokens [{start}, {end}):')
    for i in range(start, min(end, tok.get_vocab_size())):
        token = tok.id_to_token[i]
        if token in tok.special_tokens:
            surface = token
        else:
            try:
                surface = bytearray(tok.byte_decoder[c] for c in token).decode('utf-8', errors='replace')
            except (KeyError, TypeError):
                surface = token
        if i < 4:
            print(f'  id={i:5d}  {repr(token):30s}  surface={repr(surface)}')
        elif i < 260:
            if i % 32 == 4:
                print(f'  id={i:5d}  {repr(token):30s}  surface={repr(surface)}  (... base byte vocab ...)')
        else:
            print(f'  id={i:5d}  {repr(token):30s}  surface={repr(surface)}')

print('=== tokenizer_2 vocab structure ===')
show_vocab_slice(tok2, start=0, end=275)

=== tokenizer_2 vocab structure ===
Tokens [0, 275):
  id=    0  '[PAD]'                         surface='[PAD]'
  id=    1  '[UNK]'                         surface='[UNK]'
  id=    2  '[BOS]'                         surface='[BOS]'
  id=    3  '[EOS]'                         surface='[EOS]'
  id=    4  '!'                             surface='!'  (... base byte vocab ...)
  id=   36  'A'                             surface='A'  (... base byte vocab ...)
  id=   68  'a'                             surface='a'  (... base byte vocab ...)
  id=  100  '£'                             surface='�'  (... base byte vocab ...)
  id=  132  'Ä'                             surface='�'  (... base byte vocab ...)
  id=  164  'ä'                             surface='�'  (... base byte vocab ...)
  id=  196  'Ą'                             surface='\x04'  (... base byte vocab ...)
  id=  228  'Ĥ'                             surface='�'  (... base byte vocab ...)
  id=  260  'Ġt'                        

In [33]:
# Merge tree: trace how a given token was built from base bytes
def trace_token(tok, token_str):
    """Show which merges built this token string (Stage A only)."""
    # Invert bpe_ranks: rank -> pair
    by_rank = {v: k for k, v in tok.bpe_ranks.items()}

    def surf(t):
        try:
            return bytearray(tok.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
        except (KeyError, TypeError):
            return t

    def _trace(sym, depth=0):
        indent = '  ' * depth
        print(f'{indent}{repr(surf(sym))}')
        # Find the merge that produced this symbol
        for rank in range(len(by_rank)):
            a, b = by_rank[rank]
            if a + b == sym and len(sym) > 1:
                print(f'{indent}= ({repr(surf(a))} + {repr(surf(b))})  [rank {rank}]')
                _trace(a, depth + 1)
                _trace(b, depth + 1)
                return
        # Base case: single byte
        print(f'{indent}[base byte: {repr(surf(sym))}]')

    print(f'Merge tree for token {repr(surf(token_str))}:')
    _trace(token_str)

def surf2(t, tok=tok2):
    try:
        return bytearray(tok.byte_decoder[c] for c in t).decode('utf-8', errors='replace')
    except Exception:
        return t

# Pick a few interesting Stage-A tokens (medium length, all-alpha surface form)
interesting = [
    t for t in tok2.id_to_token.values()
    if t not in tok2.special_tokens
    and 4 <= len(t) <= 8
    and surf2(t).strip().isalpha()
][:3]

for tok_str in interesting:
    trace_token(tok2, tok_str)
    print()

Merge tree for token ' the':
' the'
= (' t' + 'he')  [rank 14]
  ' t'
  = (' ' + 't')  [rank 0]
    ' '
    [base byte: ' ']
    't'
    [base byte: 't']
  'he'
  = ('h' + 'e')  [rank 2]
    'h'
    [base byte: 'h']
    'e'
    [base byte: 'e']

Merge tree for token ' and':
' and'
= (' a' + 'nd')  [rank 50]
  ' a'
  = (' ' + 'a')  [rank 3]
    ' '
    [base byte: ' ']
    'a'
    [base byte: 'a']
  'nd'
  = ('n' + 'd')  [rank 26]
    'n'
    [base byte: 'n']
    'd'
    [base byte: 'd']

Merge tree for token ' The':
' The'
= (' T' + 'he')  [rank 84]
  ' T'
  = (' ' + 'T')  [rank 16]
    ' '
    [base byte: ' ']
    'T'
    [base byte: 'T']
  'he'
  = ('h' + 'e')  [rank 2]
    'h'
    [base byte: 'h']
    'e'
    [base byte: 'e']



In [34]:
# Cache efficiency demo: second encode is O(1) from cache
import time

text = domain2_dev[:2000]

# Clear cache
tok2._encode_cache.clear()
tok2.cache.clear()

t0 = time.perf_counter()
_ = tok2.encode(text)
t1 = time.perf_counter()
_ = tok2.encode(text)
t2 = time.perf_counter()

print(f'First encode  (cold cache): {(t1-t0)*1000:.2f} ms')
print(f'Second encode (warm cache): {(t2-t1)*1000:.2f} ms')
print(f'Speedup: {(t1-t0)/(t2-t1):.0f}x')

First encode  (cold cache): 1.07 ms
Second encode (warm cache): 0.02 ms
Speedup: 71x


In [35]:
# _auto_configure: what it decides for each domain
with open('data/domain_1_train.txt') as f:
    d1 = f.readlines()
with open('data/domain_2_train.txt') as f:
    d2 = f.readlines()

for name, texts in [('domain_1 (social)', d1[:500]), ('domain_2 (news)', d2[:500])]:
    step = max(1, len(texts) // 2000)
    sample = [t for t in texts[::step] if t.strip()][:2000]
    total_chars = sum(len(t) for t in sample)
    at_hash = sum(t.count('@') + t.count('#') for t in sample)
    short_frac = sum(1 for t in sample if len(t.rstrip()) < 80) / len(sample)
    at_hash_density = at_hash / total_chars if total_chars else 0
    is_social = at_hash_density > 0.005 or short_frac > 0.65
    print(f'{name}:')
    print(f'  @/# density: {at_hash_density:.4f}  short_frac: {short_frac:.3f}  is_social={is_social}')

domain_1 (social):
  @/# density: 0.0068  short_frac: 0.618  is_social=True
domain_2 (news):
  @/# density: 0.0001  short_frac: 0.606  is_social=False
